## Silver Layer – Cleaned & Validated Medical Features

### Step 1: Load Bronze Data

In [0]:
bronze_df = spark.read.table("workspace.default.bronze_heart_disease")
bronze_df.show(5)


### Step 2: Create Silver Layer (Data Cleaning & Standardization)
Clean missing values, standardize column names, and prepare medical features for analysis and ML.

In [0]:
silver_df = bronze_df \
    .withColumnRenamed("sex", "gender") \
    .withColumnRenamed("cp", "chest_pain_type") \
    .withColumnRenamed("trestbps", "resting_blood_pressure") \
    .withColumnRenamed("chol", "serum_cholesterol") \
    .withColumnRenamed("fbs", "fasting_blood_sugar") \
    .withColumnRenamed("restecg", "resting_ecg_result") \
    .withColumnRenamed("thalach", "max_heart_rate") \
    .withColumnRenamed("exang", "exercise_induced_angina") \
    .withColumnRenamed("oldpeak", "st_depression") \
    .withColumnRenamed("slope", "st_slope") \
    .withColumnRenamed("ca", "num_major_vessels") \
    .withColumnRenamed("thal", "thalassemia_type") \
    .withColumnRenamed("condition", "heart_disease_flag")

### Step 3: Transformations

In [0]:
from pyspark.sql.functions import when

# Gender
silver_df = silver_df.withColumn(
    "gender_label",
    when(silver_df.gender == 1, "Male").otherwise("Female")
)



In [0]:
# Chest Pain Type (cp)
silver_df = silver_df.withColumn(
    "chest_pain_label",
    when(silver_df.chest_pain_type == 0, "Typical Angina")
    .when(silver_df.chest_pain_type == 1, "Atypical Angina")
    .when(silver_df.chest_pain_type == 2, "Non-anginal Pain")
    .otherwise("Asymptomatic")
)



In [0]:
# Fasting Blood Sugar (fbs)
silver_df = silver_df.withColumn(
    "fasting_blood_sugar_label",
    when(silver_df.fasting_blood_sugar == 1, "High (>120 mg/dl)")
    .otherwise("Normal")
)



In [0]:
# Resting ECG (restecg)
silver_df = silver_df.withColumn(
    "resting_ecg_label",
    when(silver_df.resting_ecg_result == 0, "Normal")
    .when(silver_df.resting_ecg_result == 1, "ST-T Abnormality")
    .otherwise("Left Ventricular Hypertrophy")
)



In [0]:
# Exercise Induced Angina (exang)
silver_df = silver_df.withColumn(
    "exercise_angina_label",
    when(silver_df.exercise_induced_angina == 1, "Yes").otherwise("No")
)


In [0]:
# Thalassemia (thal)
silver_df = silver_df.withColumn(
    "thalassemia_label",
    when(silver_df.thalassemia_type == 1, "Normal")
    .when(silver_df.thalassemia_type == 2, "Fixed Defect")
    .otherwise("Reversible Defect")
)


In [0]:
# Heart Disease Target
silver_df = silver_df.withColumn(
    "heart_disease_label",
    when(silver_df.heart_disease_flag == 1, "Disease Present")
    .otherwise("No Disease")
)


In [0]:
from pyspark.sql.types import IntegerType, DoubleType

silver_df = silver_df \
    .withColumn("age", silver_df.age.cast(IntegerType())) \
    .withColumn("resting_blood_pressure", silver_df.resting_blood_pressure.cast(IntegerType())) \
    .withColumn("serum_cholesterol", silver_df.serum_cholesterol.cast(IntegerType())) \
    .withColumn("max_heart_rate", silver_df.max_heart_rate.cast(IntegerType())) \
    .withColumn("st_depression", silver_df.st_depression.cast(DoubleType()))



### Step 4: Save as Silver Table

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_heart_disease")


### step 4: Verify Silver layer data

In [0]:
%sql
SELECT * FROM workspace.default.silver_heart_disease LIMIT 10;


In [0]:
silver_df= spark.read.table("workspace.default.silver_heart_disease")
display(silver_df)